# LLM-as-a-Judge Filter

Notebook to apply a basic LLM-as-a-Judge filter on 3 datasets:
- ASQA test set (newly created by splitting the original ASQA train set)
- MS Macro (500 longest answers from the original test set)
- WikiEval 

This is to compare the performance of the filter with RAGAS filter.


## Imports and setup    

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
import json
from dotenv import load_dotenv

import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score


from IPython.display import display

sys.path.append('..')

from src.filtering.llm_judge_filter import LLMJudgeFilter



load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# Paths
# ============================================================

DATA_PATH = Path('../data/labeled_merged_test.csv')

OUTPUT_DIR = Path('../results/llm_filter')
OUTPUT_FILTER = OUTPUT_DIR / 'classification'
OUTPUT_FILTER.mkdir(parents=True, exist_ok=True)

print(f'Output dir: {OUTPUT_FILTER}')

Output dir: ..\results\llm_filter\classification


## Imports data

In [3]:
# ============================================================
# Quick schema check
# ============================================================

REQUIRED_COLS = {'id', 'question', 'answer', 'context'}
OPTIONAL_EVAL_COLS = {'label', 'gold_ans', 'gold_answer', 'reference', 'reference_answer', 'ground_truth'}

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {DATA_PATH}')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

display(df.head())

# count sample per dataset
dataset_counts = df['dataset'].value_counts()
print("Sample counts per dataset:")
display(dataset_counts)

Dataset: ..\data\labeled_merged_test.csv
Shape: (1962, 6)
Columns: ['id', 'question', 'context', 'answer', 'label', 'dataset']


,id,question,context,answer,label,dataset
0,asqa_2288_hallu,When did the first indian expedition to antarc...,- (Indian Antarctic Program) The first Indian ...,The first Indian expedition led by an Indian t...,0,asqa
1,14_pos,"When did the Inter Expo Center in Sofia, Bulga...",The Inter Expo Center (IEC) is a multi-purpose...,"The Inter Expo Center in Sofia, Bulgaria opene...",1,wikieval
2,asqa_4328,When was the first mobile phone text message s...,- (QA_1) When was the first mobile phone text ...,SMS messaging was used for the first time on 3...,1,asqa
3,asqa_3065,Who sings you'll be a woman soon?,"- (Girl, You'll Be a Woman Soon) Girl, You'll ...","Girl, You'll Be a Woman Soon is a song written...",1,asqa
4,asqa_3132,When does hotel transylvania part 3 come out?,- (Hotel Transylvania 3: Summer Vacation) Hote...,"Hotel Transylvania 3, released internationally...",1,asqa


Sample counts per dataset:


dataset
asqa        1737
msmarco      207
wikieval      18
Name: count, dtype: int64

## Initialize LLM Judge Filter & Evaluator

In [4]:
# ============================================================
# Init LLM Judge
# ============================================================

# LLMJudgeFilter sẽ đọc OPENAI_API_KEY từ environment.

judge = LLMJudgeFilter(
    model='gpt-4o-mini',
    api_key=OPENAI_API_KEY,
    output_dir=OUTPUT_FILTER,
    temperature=0.0,
    max_retries=3,
    sleep_between_retries=2.0,
    max_context_chars=12000,
)

In [7]:
# ============================================================
# Run LLM judge on one dataset
# ============================================================

def run_llm_judge_dataset(
    run_time: int,
    df: pd.DataFrame,
    eval_mode: str = 'classification',
    resume: bool = True,
    save_every: int = 1,
):
    print(f'Running LLM judge {run_time}')
    print(f'Input: {df.shape[0]} samples')

    metrics_dfs = []
    for i in range(run_time):
        print('=' * 80)
        print(f'Run {i+1}/{run_time}')
        out = judge.run(
            data=df,
            output_path=OUTPUT_FILTER / f'predictions_run_{i+1}.csv',
            eval_mode=eval_mode,
            resume=resume,
            save_every=save_every,
        )
        pred_df = pd.read_csv(OUTPUT_FILTER / f'predictions_run_{i+1}.csv')
        # Split to datasets
        metric_df = []
        for dataset_name, group in pred_df.groupby('dataset'):
            acc = accuracy_score(group['label'], group['filter_label'])
            num_samples = group.shape[0]
            accepted = (group['filter_label'] == 1).sum()
            acceptance_rate = accepted / num_samples if num_samples > 0 else 0.0
            prec = precision_score(group['label'], group['filter_label'], zero_division=0)
            red = recall_score(group['label'], group['filter_label'], zero_division=0)
            f1 = f1_score(group['label'], group['filter_label'], zero_division=0)
            roc_auc = roc_auc_score(group['label'], group['filter_label']) if len(group['label'].unique()) > 1 else float('nan')
            conf = group['filter_confidence'].mean() if 'filter_confidence' in group.columns else float('nan')      
            metric_df.append({
                "dataset": dataset_name,
                "num_samples": num_samples,
                "accuracy": acc,
                "acceptance_rate": acceptance_rate,
                "precision": prec,
                "recall": red,
                "f1_score": f1,
                "roc_auc": roc_auc,
                "avg_confidence": conf,
            }
            )
        metric_df = pd.DataFrame(metric_df)
        metrics_dfs.append(metric_df)
        display(metric_df.sort_values("dataset", ascending=True).reset_index(drop=True))

    
    # avg per dataset
    avg_metrics_df = pd.concat(metrics_dfs).groupby("dataset").mean().reset_index()
    # represent dataset name asqa -> ASQA; msmarco -> MS MARCO; etc.
    dataset_name_mapping = {
        "asqa": "ASQA",
        "msmarco": "MS MARCO",
        "wikieval": "WikiEval",
    }
    avg_metrics_df["dataset"] = avg_metrics_df["dataset"].map(dataset_name_mapping).fillna(avg_metrics_df["dataset"])

    return avg_metrics_df

## Run evaluation on Merged Test Set
Include samples fronm 3 datasets:
- ASQA
- MS MARCO
- WikiEval

In [8]:
summary_df = run_llm_judge_dataset(
    run_time=3,
    df=df,
    eval_mode='classification',
    resume=True,
    save_every=1,
)

Running LLM judge 3
Input: 1962 samples
Run 1/3
Found 1962 completed samples. Resuming from last checkpoint.
last 5 completed ids: ['asqa_912_hallu', 'asqa_1446', 'asqa_1767_hallu', 'asqa_2910', '15379']


LLM Judge (gpt-4o-mini): 0sample [00:00, ?sample/s]


,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1_score,roc_auc,avg_confidence
0,asqa,1737,0.846862,0.400115,0.935252,0.746269,0.830140,0.847153,0.915630
1,msmarco,207,0.917874,0.429952,0.988764,0.846154,0.911917,0.918223,0.897826
2,wikieval,18,0.944444,0.611111,0.909091,1.000000,0.952381,0.937500,0.916667


Run 2/3
Found 1962 completed samples. Resuming from last checkpoint.
last 5 completed ids: ['asqa_912_hallu', 'asqa_1446', 'asqa_1767_hallu', 'asqa_2910', '15379']


LLM Judge (gpt-4o-mini): 0sample [00:00, ?sample/s]


,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1_score,roc_auc,avg_confidence
0,asqa,1737,0.844560,0.398964,0.933622,0.742824,0.827366,0.844853,0.916436
1,msmarco,207,0.922705,0.434783,0.988889,0.855769,0.917526,0.923030,0.898792
2,wikieval,18,0.888889,0.666667,0.833333,1.000000,0.909091,0.875000,0.925000


Run 3/3
Found 1962 completed samples. Resuming from last checkpoint.
last 5 completed ids: ['asqa_912_hallu', 'asqa_1446', 'asqa_1767_hallu', 'asqa_2910', '15379']


LLM Judge (gpt-4o-mini): 0sample [00:00, ?sample/s]


,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1_score,roc_auc,avg_confidence
0,asqa,1737,0.846287,0.397237,0.937681,0.742824,0.828956,0.846585,0.915774
1,msmarco,207,0.913043,0.425121,0.988636,0.836538,0.906250,0.913415,0.899758
2,wikieval,18,0.888889,0.666667,0.833333,1.000000,0.909091,0.875000,0.925000


In [9]:
display(summary_df.sort_values("dataset", ascending=True).reset_index(drop=True))

summary_df.to_csv(OUTPUT_FILTER / 'llm_judge_summary.csv', index=False)
print(f"Saved average dataset metrics to: {OUTPUT_FILTER / 'llm_judge_summary.csv'}")

,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1_score,roc_auc,avg_confidence
0,ASQA,1737.0,0.845903,0.398772,0.935518,0.743972,0.828821,0.846197,0.915947
1,MS MARCO,207.0,0.917874,0.429952,0.988763,0.846154,0.911898,0.918223,0.898792
2,WikiEval,18.0,0.907407,0.648148,0.858586,1.000000,0.923521,0.895833,0.922222


Saved average dataset metrics to: ..\results\llm_filter\classification\llm_judge_summary.csv
